<a href="https://colab.research.google.com/github/Nipun1a/Credit_predict/blob/main/credit_card_crc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
```

Importing the dependencies


In [51]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [52]:
# Loading the dataset to a pandas dataframe
credit_card_data = pd.read_csv('/content/credit_risk_dataset.csv')

In [53]:
#first 5 rows of the dataset
credit_card_data.head()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Education_Level,Housing_Status,Default
0,59,52154.0,11276,823,15,Bachelors,Own,0
1,49,116646.0,43663,315,5,PhD,Own,0
2,35,61157.0,18994,428,8,Masters,Own,1
3,63,52154.0,28499,408,26,Bachelors,Rent,0
4,28,148876.0,28040,832,3,Masters,Own,1


In [54]:
credit_card_data.tail()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Education_Level,Housing_Status,Default
995,53,44519.0,7307,433,22,PhD,Rent,1
996,22,107487.0,44901,582,7,High School,Own,1
997,34,102870.0,16205,372,29,Masters,Rent,0
998,60,66197.0,10906,780,24,PhD,Own,0
999,60,NaN,7591,651,5,High School,Mortgage,0


In [55]:
#dataset information
credit_card_data.info()

# The NameError for 'legit' indicates that the variable 'legit' has not been defined.
# Please ensure that the cell defining 'legit' and 'fraud' (cell RWxqh4OAT_B5) has been executed.
# The relevant code for defining 'legit' and 'fraud' is:
# legit = credit_card_data[credit_card_data.Class == 0]
# fraud = credit_card_data[credit_card_data.Class == 1]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               1000 non-null   int64  
 1   Income            985 non-null    float64
 2   Loan_Amount       1000 non-null   int64  
 3   Credit_Score      1000 non-null   int64  
 4   Employment_Years  1000 non-null   int64  
 5   Education_Level   1000 non-null   object 
 6   Housing_Status    1000 non-null   object 
 7   Default           1000 non-null   int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 62.6+ KB


In [56]:
# checking the number of missing values in each column
credit_card_data.isnull().sum()

# Fill missing 'Income' values with the mean of the column
credit_card_data['Income'] = credit_card_data['Income'].fillna(credit_card_data['Income'].mean())

In [57]:
# distribution of legit transaction & fraudulent transactions
credit_card_data['Default'].value_counts()

,count
Default,
0,862
1,138


This dataset is highly unbalanced

0 --> Normal Trandaction

1 --> fradulent Transaction

In [58]:
#seprating the data for analysis
legit = credit_card_data[credit_card_data.Default == 0]
fraud = credit_card_data[credit_card_data.Default == 1]

In [59]:
print(legit.shape)
print(fraud.shape)

(862, 8)
(138, 8)


In [60]:
# statistical measures of the data
legit.Loan_Amount.describe()

,Loan_Amount
count,862.000000
mean,27965.696056
std,12800.694225
min,5097.000000
25%,16537.250000
50%,28632.000000
75%,38926.000000
max,49976.000000


In [61]:
fraud.Loan_Amount.describe()

,Loan_Amount
count,138.000000
mean,26252.855072
std,12557.228545
min,5281.000000
25%,15882.500000
50%,25701.000000
75%,36128.000000
max,49298.000000


In [62]:
# compare the values for both transactions
credit_card_data.groupby('Default').mean(numeric_only=True)

,Age,Income,Loan_Amount,Credit_Score,Employment_Years
Default,,,,,
0,42.976798,82402.181639,27965.696056,580.453596,15.220418
1,39.630435,81328.747414,26252.855072,584.775362,14.811594


Under-Sampling

Build a sample dataset containing similar distribution of normal transactions and Fraudulent Transaction


Number of Fradulent Transactions --> 492

In [63]:
legit_sample = legit.sample(n=492)

Concatenating Two dataFrames

In [64]:
new_dataset = pd.concat([legit_sample, fraud], axis = 0)

# One-Hot Encode categorical features for the new_dataset
new_dataset = pd.get_dummies(new_dataset, columns=['Education_Level', 'Housing_Status'], drop_first=True)

# Check for NaNs in new_dataset after all transformations
print("NaNs in new_dataset after one-hot encoding:\n", new_dataset.isnull().sum())

# Impute any remaining numeric NaNs in new_dataset, if any
for col in new_dataset.select_dtypes(include=np.number).columns:
    if new_dataset[col].isnull().any():
        new_dataset[col] = new_dataset[col].fillna(new_dataset[col].mean())

NaNs in new_dataset after one-hot encoding:
 Age                            0
Income                         0
Loan_Amount                    0
Credit_Score                   0
Employment_Years               0
Default                        0
Education_Level_High School    0
Education_Level_Masters        0
Education_Level_PhD            0
Housing_Status_Own             0
Housing_Status_Rent            0
dtype: int64


In [65]:
new_dataset.head()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Default,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent
937,22,145733.0,11823,358,12,0,False,False,False,True,False
563,49,39029.0,43278,737,3,0,False,True,False,True,False
582,57,49486.0,37125,410,27,0,False,False,True,False,True
116,64,29446.0,9114,361,12,0,True,False,False,False,False
739,31,84435.0,48922,472,28,0,False,True,False,False,False


In [66]:
new_dataset.tail()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Default,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent
940,57,107487.0,34118,349,11,1,True,False,False,False,True
983,28,52265.0,5282,594,29,1,False,False,False,True,False
986,34,148876.0,9135,529,2,1,False,True,False,True,False
995,53,44519.0,7307,433,22,1,False,False,True,False,True
996,22,107487.0,44901,582,7,1,True,False,False,True,False


In [67]:
new_dataset['Default'].value_counts()

,count
Default,
0,492
1,138


In [68]:
new_dataset.groupby('Default').mean(numeric_only=True)

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent
Default,,,,,,,,,,
0,42.774390,81320.886761,28096.095528,580.436992,15.254065,0.237805,0.247967,0.250000,0.327236,0.308943
1,39.630435,81328.747414,26252.855072,584.775362,14.811594,0.202899,0.282609,0.275362,0.326087,0.326087


Splitting the data into Features & Targets

In [69]:
X = new_dataset.drop(columns='Default', axis=1)
Y = new_dataset['Default']

In [70]:
print(X)

     Age    Income  Loan_Amount  Credit_Score  Employment_Years  \
937   22  145733.0        11823           358                12   
563   49   39029.0        43278           737                 3   
582   57   49486.0        37125           410                27   
116   64   29446.0         9114           361                12   
739   31   84435.0        48922           472                28   
..   ...       ...          ...           ...               ...   
940   57  107487.0        34118           349                11   
983   28   52265.0         5282           594                29   
986   34  148876.0         9135           529                 2   
995   53   44519.0         7307           433                22   
996   22  107487.0        44901           582                 7   

     Education_Level_High School  Education_Level_Masters  \
937                        False                    False   
563                        False                     True   
582         

In [71]:
print(Y)

937    0
563    0
582    0
116    0
739    0
      ..
940    1
983    1
986    1
995    1
996    1
Name: Default, Length: 630, dtype: int64


Split the data into Training data & Testing data

In [72]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [73]:
print(X.shape, X_train.shape, X_test.shape)

(630, 10) (504, 10) (126, 10)


Model Training

Logistic Regression

In [74]:
model = LogisticRegression(solver='liblinear', max_iter=200, random_state=42)

In [75]:
# training the logistic Regression Model with Training Data
print("Checking for NaNs in X_train before fitting:")
print(X_train.isnull().sum())
print("Total NaNs in X_train:", X_train.isnull().sum().sum())
model.fit(X_train, Y_train)

Checking for NaNs in X_train before fitting:
Age                            0
Income                         0
Loan_Amount                    0
Credit_Score                   0
Employment_Years               0
Education_Level_High School    0
Education_Level_Masters        0
Education_Level_PhD            0
Housing_Status_Own             0
Housing_Status_Rent            0
dtype: int64
Total NaNs in X_train: 0


LogisticRegression(max_iter=200, random_state=42, solver='liblinear')

Model Evaluation

Accuracy score

In [76]:
# accuracy on training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [77]:
print('Accuracy on Training data:', training_data_accuracy)

Accuracy on Training data: 0.7817460317460317


In [78]:
#acuracy on test data
print("Checking for NaNs in X_test before prediction:")
print(X_test.isnull().sum())
print("Total NaNs in X_test:", X_test.isnull().sum().sum())
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

Checking for NaNs in X_test before prediction:
Age                            0
Income                         0
Loan_Amount                    0
Credit_Score                   0
Employment_Years               0
Education_Level_High School    0
Education_Level_Masters        0
Education_Level_PhD            0
Housing_Status_Own             0
Housing_Status_Rent            0
dtype: int64
Total NaNs in X_test: 0


In [79]:
print('Accuracy score on Test Data:', test_data_accuracy)

Accuracy score on Test Data: 0.7777777777777778


### Confusion Matrix

A confusion matrix is a table that is used to evaluate the performance of a classification model. It summarizes the number of correct and incorrect predictions made by the classifier with respect to the actual outcomes. It has four key components:

*   **True Positives (TP):** Correctly predicted positive values.
*   **True Negatives (TN):** Correctly predicted negative values.
*   **False Positives (FP):** Incorrectly predicted positive values (Type I error).
*   **False Negatives (FN):** Incorrectly predicted negative values (Type II error).

In [80]:
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
conf_matrix = confusion_matrix(Y_test, X_test_prediction)

print('Confusion Matrix:')
print(conf_matrix)

Confusion Matrix:
[[98  0]
 [28  0]]
